# LAB 01 — Observando tokens

**Objetivo:** relacionar **Texto → Tokens → Contexto → Costo**.

**Tiempo sugerido:** 25 minutos · **No requiere API key** (excepto la sección opcional).

Al terminar podrás responder:
1. ¿Cuántos tokens utiliza cada tipo de texto?
2. ¿Qué diferencias encuentras entre español, inglés y código?
3. ¿Cómo afecta esto al costo y a la latencia de una aplicación real?

In [ ]:
# Verificación del entorno: este notebook debe correr con el kernel .venv del repositorio
import sys, importlib.util
_faltan = [p for p in ["numpy", "pandas", "sklearn", "tiktoken", "openai", "dotenv", "matplotlib"]
           if importlib.util.find_spec(p) is None]
if _faltan:
    raise ModuleNotFoundError(
        f"Faltan paquetes: {', '.join(_faltan)}\n"
        f"Este kernel usa: {sys.executable}\n"
        "Solución: en la terminal, dentro del repositorio, ejecuta `uv sync`; luego en VS Code elige "
        "Select Kernel → Python Environments → .venv (la del repositorio) y reinicia el kernel."
    )
print("✅ Entorno listo:", sys.executable)

In [ ]:
# Configuración común: localiza el repositorio, carga .env y el documento de trabajo
import os
import json
from pathlib import Path
from dotenv import load_dotenv

def encontrar_raiz():
    p = Path.cwd().resolve()
    for c in [p, *p.parents]:
        if (c / "data").is_dir() and (c / "pyproject.toml").exists():
            return c
    raise FileNotFoundError("No encontré la carpeta data/. Abre el notebook desde el repositorio del curso.")

ROOT = encontrar_raiz()
load_dotenv(ROOT / ".env")
DATA_DIR = ROOT / "data"

DOCUMENTOS = {
    "cafeteria":   ("cafeteria_nebula_manual.md",      "cafeteria_preguntas.json"),
    "streamly":    ("streamly_api_docs.md",            "streamly_preguntas.json"),
    "veterinaria": ("clinica_veterinaria_huellitas.md", "veterinaria_preguntas.json"),
    "mi_documento": ("mi_documento.md",                 None),  # genera el tuyo con los prompts de alumnos/02_material_del_alumno.md
}

def cargar_documento(nombre):
    archivo, _ = DOCUMENTOS[nombre]
    return (DATA_DIR / archivo).read_text(encoding="utf-8")

def cargar_preguntas(nombre):
    _, archivo = DOCUMENTOS[nombre]
    if archivo is None:
        return []
    return json.loads((DATA_DIR / archivo).read_text(encoding="utf-8"))

print("Repositorio:", ROOT)

In [ ]:
# 👉 Elige el documento con el que vas a trabajar
DOCUMENTO = "cafeteria"   # "cafeteria" | "streamly" | "veterinaria" | "mi_documento"

texto = cargar_documento(DOCUMENTO)
print(f"Documento: {DOCUMENTO} — {len(texto):,} caracteres, {len(texto.split()):,} palabras")
print(texto[:500], "...")

## 1. Un tokenizador

Usamos `tiktoken`, el tokenizador de OpenAI. `o200k_base` es la codificación de los modelos recientes de OpenAI
y `cl100k_base` la de una generación anterior. Otros proveedores (y los modelos de Ollama) usan tokenizadores distintos,
así que **los números cambian entre modelos**: lo importante es el orden de magnitud y las diferencias entre textos.

In [ ]:
import tiktoken
import pandas as pd

enc = tiktoken.get_encoding("o200k_base")
enc_anterior = tiktoken.get_encoding("cl100k_base")

frase = "Construiremos un RAG desde cero."
ids = enc.encode(frase)
print(ids)

### ✏️ Ejercicio 1 — Cuenta y visualiza los tokens

Completa las dos funciones:
- `contar_tokens(texto, enc)` → número de tokens.
- `ver_tokens(texto, enc)` → lista con el texto de cada token (pista: `enc.decode([id])`).

In [ ]:
def contar_tokens(texto, enc):
    # TODO: devuelve cuántos tokens tiene el texto
    ...

def ver_tokens(texto, enc):
    # TODO: devuelve una lista con el fragmento de texto que representa cada token
    ...

print(contar_tokens(frase, enc))
print(ver_tokens(frase, enc))

In [ ]:
# ✅ Autoverificación
assert contar_tokens(frase, enc) == len(enc.encode(frase))
assert "".join(ver_tokens(frase, enc)) == frase
print("✅ Ejercicio 1 correcto")

## 2. Comparando tipos de texto

### ✏️ Ejercicio 2 — Construye la tabla

Completa el diccionario `textos` (ya tiene ejemplos, puedes cambiarlos) y construye un DataFrame con las columnas:
`texto`, `caracteres`, `palabras`, `tokens`, `tokens_por_palabra`, `tokens_cl100k`.

In [ ]:
textos = {
    "frase corta": "La inteligencia artificial generativa puede crear texto, código e imágenes.",
    "párrafo": texto[texto.find("\n\n", 400):][:700].strip(),
    "documento técnico (completo)": texto,
    "código": '''def cosine(a, b):
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    return float(np.dot(a, b) / denom) if denom else 0.0''',
    "español": "¿Cuál es el horario de atención del equipo de soporte los fines de semana?",
    "inglés": "What are the support team's opening hours on weekends?",
}

In [ ]:
filas = []
for nombre, t in textos.items():
    # TODO: agrega un diccionario por texto con las columnas indicadas
    ...

tabla = pd.DataFrame(filas)
tabla

In [ ]:
# ✅ Autoverificación
assert set(["texto", "caracteres", "palabras", "tokens", "tokens_por_palabra", "tokens_cl100k"]) <= set(tabla.columns)
assert len(tabla) == len(textos)
print("✅ Ejercicio 2 correcto")

In [ ]:
# Visualiza cómo se parte la pregunta en español vs. inglés
for nombre in ["español", "inglés"]:
    print(nombre, "→", ver_tokens(textos[nombre], enc))

**Observa:** ¿qué texto tiene más tokens por palabra? ¿el español o el inglés? ¿y el código?
Escribe tu observación aquí:

> ...

## 3. Tokens → costo

Los proveedores cobran por **millón de tokens** (entrada y salida por separado).
Consulta el precio vigente del modelo que uses en la página de precios del proveedor y escríbelo abajo.
*(No lo dejamos fijo porque los precios cambian.)*

### ✏️ Ejercicio 3 — ¿Mandar el documento completo o solo lo relevante?

Imagina un chatbot que recibe **1,000 preguntas al día**. Compara dos estrategias:
- **A)** enviar el documento completo en cada pregunta.
- **B)** enviar solo 3 fragmentos de ~300 tokens (lo que hará nuestro RAG).

Completa `costo_diario(tokens_por_pregunta, preguntas_por_dia, precio_por_millon)`.

In [ ]:
PRECIO_POR_MILLON_ENTRADA = None   # TODO: escribe el precio en USD por 1M tokens de entrada de tu modelo

def costo_diario(tokens_por_pregunta, preguntas_por_dia, precio_por_millon):
    # TODO
    ...

tokens_doc = contar_tokens(texto, enc)
tokens_rag = 3 * 300

if PRECIO_POR_MILLON_ENTRADA is not None:
    print("A) documento completo:", costo_diario(tokens_doc, 1000, PRECIO_POR_MILLON_ENTRADA), "USD/día")
    print("B) RAG con 3 chunks  :", costo_diario(tokens_rag, 1000, PRECIO_POR_MILLON_ENTRADA), "USD/día")
print(f"El documento completo usa {tokens_doc / tokens_rag:.1f}x más tokens de entrada que la estrategia RAG")

In [ ]:
# ✅ Autoverificación
assert abs(costo_diario(1_000_000, 1, 2.0) - 2.0) < 1e-9
assert abs(costo_diario(500, 1000, 1.0) - 0.5) < 1e-9
print("✅ Ejercicio 3 correcto")

## 4. (Opcional) Tokens reales que reporta la API

Las respuestas de la API incluyen `usage` con los tokens que realmente se cobraron.
Compara contra tu conteo con `tiktoken` (no coinciden exactamente: el formato de mensajes agrega algunos tokens,
y si usas Ollama el tokenizador es otro).

In [ ]:
EJECUTAR_OPCIONAL = False   # cambia a True si ya configuraste tu proveedor

if EJECUTAR_OPCIONAL:
    from openai import OpenAI
    proveedor = os.getenv("LLM_PROVEEDOR", "openai")
    if proveedor == "ollama":
        client = OpenAI(base_url=os.getenv("OLLAMA_BASE_URL", "http://localhost:11434/v1"), api_key="ollama")
        modelo = os.getenv("OLLAMA_CHAT_MODEL", "llama3.2")
    else:
        client, modelo = OpenAI(), "gpt-4.1-mini"
    pregunta = textos["español"]
    r = client.chat.completions.create(model=modelo, messages=[{"role": "user", "content": pregunta}], max_tokens=50)
    print("tiktoken:", contar_tokens(pregunta, enc), "| API prompt_tokens:", r.usage.prompt_tokens,
          "| completion_tokens:", r.usage.completion_tokens)

## Preguntas de cierre (discútelas con tu equipo)

1. ¿Cuántos tokens tiene tu documento completo? ¿Cabría en la ventana de contexto de un modelo pequeño?
2. ¿Qué diferencias estructurales encontraste entre español, inglés y código?
3. Si tu chatbot recibe 1,000 preguntas al día, ¿qué estrategia elegirías y por qué?
4. Además del costo, ¿qué otra métrica empeora al enviar más tokens? *(pista: latencia)*